<a href="https://colab.research.google.com/github/oni-swr/bvh-mocap-via-video/blob/collab/eccv22_demo/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import tensorflow_hub as hub
from pathlib import Path
import math
import imageio as iio
from google.colab import drive
drive.mount('/content/drive')
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

path='/content/drive/MyDrive/motion capture /unpacked'
#p = Path(path)
#if p.exists():
#    print(p.read_text())
model = tf.saved_model.load(path)  # Takes about 3 minutes






Mounted at /content/drive
Found GPU at: /device:GPU:0


In [3]:
video_path='/content/drive/MyDrive/videos/2025-10-29 11-03-28.mkv'

In [4]:
def frame_gen(path):
    for frame in iio.imiter(path):  # HxWx3 uint8 RGB
        yield frame

def resize_to_384(img):
    img = tf.image.resize_with_pad(img, 384, 384, method="bilinear")
    return tf.cast(tf.round(img), tf.uint8)

ds = tf.data.Dataset.from_generator(
    lambda: frame_gen(video_path),
    output_signature=tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
).map(resize_to_384, num_parallel_calls=tf.data.AUTOTUNE)

batch_size = 32  # try 16/32/64; increase until you hit GPU OOM, then back off
frame_batches = ds.batch(batch_size, drop_remainder=False).cache().prefetch(tf.data.AUTOTUNE)

In [5]:
H = W = 384
fov_deg = 55.0
fx = fy = 0.5 * W / math.tan(0.5 * math.radians(fov_deg))
cx, cy = W / 2.0, H / 2.0
K = tf.constant([[fx, 0.0, cx],
                 [0.0, fy, cy],
                 [0.0, 0.0, 1.0]], dtype=tf.float32)  # [3,3]

In [6]:
import imageio as iio

In [7]:
skeleton = "smpl+head_30"

@tf.function  # keep jit_compile=False by default; XLA can help on some GPUs but benchmark first
def run_batch(images):
    B = tf.shape(images)[0]
    K_batched = tf.repeat(K[tf.newaxis, :, :], repeats=B, axis=0)  # [B,3,3]
    return model.detect_poses_batched(
        images,
        skeleton=skeleton

    )



In [14]:

for i,batch in enumerate(frame_batches):
    with tf.device("/GPU:0"):  # ensures kernels run on the GPU when available
        pred = run_batch(batch)
        print(f"Processed frames:{pred['poses3d'].shape}")
    poses3d = pred["poses3d"].numpy() # Ragged [B, (num_persons_i), J, 3]
    np.save(f"poses_batch_{i:04d}.npy", poses3d)


Processed frames:(32, None, 30, 3)
Processed frames:(32, None, 30, 3)
Processed frames:(32, None, 30, 3)
Processed frames:(32, None, 30, 3)
Processed frames:(32, None, 30, 3)
Processed frames:(32, None, 30, 3)
Processed frames:(32, None, 30, 3)
Processed frames:(32, None, 30, 3)
Processed frames:(23, None, 30, 3)
Processed batches: 0


In [9]:
import numpy as np

np.save('/content/drive/MyDrive/motion capture /results',all_poses)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (9,) + inhomogeneous part.

In [ ]:
joint_names = model.per_skeleton_joint_names['smpl+head_30'].numpy().astype(str)
joint_edges = model.per_skeleton_joint_edges['smpl+head_30'].numpy()


In [ ]:
np.save('/content/drive/MyDrive/motion capture /joint_names',joint_names)
np.save('/content/drive/MyDrive/motion capture /joint_edges',joint_edges)